# 02 · Motor de texto con BETO

**Fase 3 — Semanas 9 a 16. El núcleo del aporte. Corre en Google Colab con GPU.**

Aquí el idioma español importa y tu corpus boliviano se vuelve indispensable.

Pasos (según el PDF):
1. Repetir la **línea base clásica** ahora sobre el corpus en español.
2. **Ajuste fino de BETO**: 3–4 épocas, tasa de aprendizaje ~`2e-5`.
3. Vigilar el **desbalance**: ponderar la función de pérdida en vez de descartar ejemplos.
4. Si el corpus queda corto, aumentar datos con variaciones controladas y declararlo.

**Entregable:** este cuaderno, modelo ajustado y comparación contra la línea base en español.

> ⚠️ En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** antes de entrenar.

## 0. Preparación (solo en Colab)

Descomenta estas celdas la primera vez en Colab.

In [ ]:
# Instalar dependencias (Colab ya trae torch)
# !pip -q install -U transformers accelerate

# Traer el corpus clonando tu repo (el corpus.csv anonimizado SÍ está versionado)
# !git clone https://github.com/Ortunex1/Modelotest.git
print('Si estás en Colab, descomenta las líneas de arriba y ejecútalas una vez.')

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Sin GPU: activa GPU en Colab (Entorno de ejecución → Cambiar tipo).')

## 1. Cargar el corpus y particionar

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Busca corpus.csv en el repo (local o clonado en Colab)
candidatos = [
    Path.cwd().parent / 'data' / 'anonimizado' / 'corpus.csv',   # local (notebooks/)
    Path('Modelotest/data/anonimizado/corpus.csv'),              # Colab tras git clone
    Path('data/anonimizado/corpus.csv'),
]
ruta = next((p for p in candidatos if p.exists()), None)

if ruta is not None:
    df = pd.read_csv(ruta)
    df = df[['texto', 'etiqueta']].dropna()
    print('Corpus real:', ruta, '| filas:', len(df))

if ruta is None or len(df) < 40:
    print('Corpus insuficiente -> usando DEMO en español (placeholder) para probar el flujo.')
    fraude = [
        'Su cuenta sera bloqueada, ingrese aqui para validar sus datos',
        'Felicidades gano un premio, confirme sus datos para cobrar',
        'Alerta: movimiento sospechoso, verifique su clave ahora',
        'Su paquete esta retenido, pague la aduana en este enlace',
        'Oferta de empleo, envie su deposito para el registro',
    ] * 12
    legit = [
        'Hola, te confirmo la reunion de manana a las diez',
        'Recordatorio de tu cita medica el viernes',
        'Gracias por tu compra, tu pedido llega el lunes',
        'Buenos dias, adjunto el informe que pediste',
        'Tu factura del mes ya esta disponible en la app',
    ] * 12
    df = pd.DataFrame({'texto': fraude + legit,
                       'etiqueta': ['fraudulento']*len(fraude) + ['legitimo']*len(legit)})
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(df['etiqueta'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train, resto = train_test_split(df, test_size=0.30, stratify=df['etiqueta'], random_state=SEED)
val, test = train_test_split(resto, test_size=0.50, stratify=resto['etiqueta'], random_state=SEED)
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

## 2. Línea base clásica en español (el rival)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score

base = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(max_iter=1000, class_weight='balanced'),
)
base.fit(train['texto'], train['etiqueta'])
pred_base = base.predict(val['texto'])
f1_base = f1_score(val['etiqueta'], pred_base, pos_label='fraudulento', zero_division=0)
print(f'Baseline clásico — F1 (fraudulento) = {f1_base:.3f}')
print(classification_report(val['etiqueta'], pred_base, digits=3, zero_division=0))

## 3. Ajuste fino de BETO

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from torch.utils.data import Dataset
from collections import Counter

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'  # BETO
ETIQUETAS = ['legitimo', 'fraudulento']
lab2id = {l: i for i, l in enumerate(ETIQUETAS)}

tok = AutoTokenizer.from_pretrained(CKPT)

class DatasetTexto(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tok(list(textos), truncation=True, padding=True, max_length=128)
        self.labels = [lab2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item

ds_tr = DatasetTexto(train['texto'], train['etiqueta'])
ds_val = DatasetTexto(val['texto'], val['etiqueta'])

In [ ]:
# Pesos por clase para la pérdida (Paso 3 del PDF: ponderar en vez de descartar)
cnt = Counter(train['etiqueta'])
total = sum(cnt.values())
pesos = torch.tensor([total / (2 * cnt[l]) for l in ETIQUETAS], dtype=torch.float)
print('Conteo por clase:', dict(cnt))
print('Pesos:', {l: round(float(p), 3) for l, p in zip(ETIQUETAS, pesos)})

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=pesos.to(model.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = pred.label_ids
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1],
                                                  average=None, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_fraudulento': f1[1],
            'recall_fraudulento': r[1],
            'precision_fraudulento': p[1]}

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=2, id2label={i: l for l, i in lab2id.items()}, label2id=lab2id)

args = TrainingArguments(
    output_dir='../modelos/beto_ckpt',
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',        # en transformers viejos: evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_fraudulento',
    seed=SEED,
    logging_steps=20,
    report_to='none',
)

trainer = TrainerPonderado(model=modelo, args=args,
                           train_dataset=ds_tr, eval_dataset=ds_val,
                           compute_metrics=metricas)
trainer.train()

## 4. Evaluación y comparación con la línea base

In [ ]:
res = trainer.evaluate()
f1_beto = res['eval_f1_fraudulento']
print(f'BETO    — F1 (fraudulento) = {f1_beto:.3f}')
print(f'Baseline— F1 (fraudulento) = {f1_base:.3f}')
print('\nBETO gana al baseline' if f1_beto > f1_base else '\nRevisar: el baseline no fue superado')
print('\n(El veredicto formal de la hipótesis se hace con la prueba de McNemar en 03_evaluacion, sobre el TEST.)')

In [ ]:
# Guardar el modelo ajustado y las predicciones de validación (para la Fase 4)
trainer.save_model('../modelos/beto_final')
tok.save_pretrained('../modelos/beto_final')

logits = trainer.predict(ds_val).predictions
logits = logits[0] if isinstance(logits, tuple) else logits
pred_beto = [ETIQUETAS[i] for i in np.argmax(logits, axis=1)]
pd.DataFrame({'texto': val['texto'].values, 'y_real': val['etiqueta'].values,
              'pred_beto': pred_beto, 'pred_baseline': pred_base}).to_csv(
    '../resultados/02_texto_val_pred.csv', index=False)
print('Modelo en modelos/beto_final y predicciones en resultados/ ✔')
print('En Colab, copia modelos/beto_final a tu Google Drive para no perderlo al cerrar la sesión.')

## Qué sigue

- El **test** sigue sin abrirse: se usa recién en `03_evaluacion.ipynb` (Fase 4) con la **prueba de McNemar** entre BETO y el baseline, y el criterio **F1 ≥ 0.90**.
- Si el corpus es pequeño y BETO sobreajusta (F1 alto en train, bajo en val), reduce épocas, sube el peso de la clase minoritaria o aumenta datos con variaciones controladas (y decláralo en la tesis).